## M1

**1. Which function takes most total time?** <br>
<br>
The function with the most total time is the naive implementation with loops<br>
<br>
**2. Are there functions called surprisingly many times?** <br>
<br>
In the implementation without numpy there are a lot of calls to built-in methods, in this case it has checked for the magnitude of values millions of times.<br>
<br>
**3. How does NumPy profile compare to naive?** <br>
<br>
The operations are no longer run by python in the case of the numpy implementation, therefore insterad of seeing millions of operations usingbuilt-in functions from python we observe numpy C functions being called an equal ammount to our loops.<br>
<br>
**4. Where does NumPy spend its time?** <br>
<br>
Almost all the runtime happens inside C-level numpy code.<br>
<br>

## M2

1. cProfile on naive vs NumPy: How many functions appear in each profile? What does
this difference tell you about where the work actually happens?<br>
<br>
The work in the naive implementation happens at python level, where the profiler is able to see the functions as built-in functions. The numpy implementation does not see the millions of element wise operations as they happen in C, below python.<br>

**NAIVE PROFILER**
```
Mon Mar  2 18:16:43 2026    naive_profile.prof

         20943329 function calls in 16.691 seconds

   Ordered by: cumulative time

    ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000   16.691   16.691 {built-in method builtins.exec}
        1    0.001    0.001   16.691   16.691 <string>:1(<module>)
        1   14.029   14.029   16.690   16.690 C:\Users\Aran\AppData\Local\Temp\ipykernel_27848\774291089.py:1(mandelbrot_naive)
        20943298    2.661    0.000    2.661    0.000 {built-in method builtins.abs}
```
**NUMPY PROFILER**
```
    Mon Mar  2 18:23:41 2026    numpy_profile.prof

        399 function calls in 1.063 seconds

    Ordered by: cumulative time

    ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    1.063    1.063 {built-in method builtins.exec}
        1    0.002    0.002    1.063    1.063 <string>:1(<module>)
        1    1.050    1.050    1.061    1.061 C:\Users\Aran\AppData\Local\Temp\ipykernel_27848\414567218.py:1(mandelbrot_numpy)
        1    0.005    0.005    0.005    0.005 c:\Python311\Lib\site-packages\numpy\_core\numeric.py:64(zeros_like)
        100    0.001    0.000    0.005    0.000 {method 'any' of 'numpy.ndarray' objects}
        100    0.000    0.000    0.004    0.000 c:\Python311\Lib\site-packages\numpy\_core\_methods.py:58(_any)
        100    0.004    0.000    0.004    0.000 {method 'reduce' of 'numpy.ufunc' objects}
        1    0.000    0.000    0.001    0.001 {built-in method builtins.print}
```
2. line profiler on naive: Which lines dominate runtime? What fraction of total time is
spent in the inner loop?<br>
<br>
The lines that dominate runtime are z = z**2 + screen[i, j] and if abs(z) > 2, these lines account for 79% of the total runtime, with an aproximate grand total of 95% of the time inside of the loop. <br>
```
    Line #    Hits        Time      Per Hit   % Time                Line Contents
    ==================================================================================
        11  12456060   22057977.0      1.8     15.0              for k in range(max_iter):
        12  12279956   70972627.0      5.8     48.2                 z = z**2 + screen[i, j]
        13  12279956   45454329.0      3.7     30.9                 if abs(z) > 2:
        14    823896    1260654.0      1.5      0.9                     break
```
3. Based on your profiling results: why is NumPy faster than naive Python?<br>
<br>
Python has to realize millions of operations whereas numpy optimizes these into a few hundred C-level optimized array operations.<br>
<br>
4. What would you need to change to make the naive version faster?<br>
<br>
Some operations can be improved by breaking them down into operations that are easyer to calculate by python such as changing Z**2 to Z*Z.<br>
<br>

# M3

```
Naive: 10.839s
NumPy: 1.010s (10.7x)
Numba: 0.060s (181.6x)
```

## M4

▶ Speed: Does float32 actually run faster than float64 on your hardware? By how much?<br>
<br>
They take the same time:<br>
float32: 0.065s<br>
float64: 0.065s<br>
<br>
▶ float16: Try it with NumPy — is it faster than float32? (It may not be, since NumPy
promotes to float32 internally.) Numba may refuse to compile float16 — if so, skip it and
note why.<br>
<br>
I do not understand the question, Numpy is not faster than mamba in any case.<br>
<br>
▶ Visual quality: Zoom in on a detailed region of the Mandelbrot set. Can you see
artefacts with float16? What about float32?<br>
<br>
I was unable to find any artifacts visible to the naked eye on both types, I've tried to increase resolution and threshold but none appear.<br>
<br>
▶ Recommendation: Based on what you observe, which precision would you choose for
production use, and why?<br>
<br>
In this case the decision lies on a trade-off between precision and memory, as I was unable to see any artifact I would probably choose float 32 for memory resons. 
<br>

## M5

Experiment 1: @jit vs. @njit <br>

```
Numba: 0.062s
Njit: 0.062s
```
There does not seem to be an improvement in this case.<br>
<br>
Experiment 2: Parallel Numba (advanced)<br>
<br>
Speedup seems considerable when moving from non parallel code to parallel code.<br>
```
t_njit: 0.504s
Parallel: 0.097s
Speedup = 5.2039943657766345
```
Experiment 3: Scaling with grid size <br>
<br>
Speedup seems to increase as the resolution goes up for the parallel implementation, the others seem to more or less stay the same.<br>
```
Naive: 2.707s
NumPy: 0.173s (15.6x)
Numba: 0.016s (168.2x)
Njit: 0.016s (170.5x)
Parallel: 0.003s (862.8x)
Naive: 10.545s
NumPy: 1.031s (10.2x)
Numba: 0.062s (169.0x)
Njit: 0.062s (170.3x)
Parallel: 0.011s (941.8x)
Naive: 42.477s
NumPy: 4.212s (10.1x)
Numba: 0.249s (170.5x)
Njit: 0.248s (171.3x)
Parallel: 0.045s (948.1x)
```